In [137]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import math

In [138]:
FILE = './output.csv'

"""
raw df has continous time series data for each user, no gap between days.
1 sugg.select.utime is Nan: invalid
Number of unique users: 37
"""
df = pd.read_csv(FILE)


In [139]:
cols_dropped = ['sugg.select.utime', 'sugg.response.utime', 'dec.location.category']

sdf = df.drop(columns=cols_dropped)

print(sdf.columns)

cols = ['uid', 'decision_idx', 'datetime', 'date', 'day_slot', 'is_randomized', 'avail', 'send', 'send_active', 'send_sedentary', 'returned_message', 'response', 'activity', 'location', 'weather', 'temperature', 'jbsteps10', 'jbsteps30', 'jbsteps40', 'jbsteps60', 'jbsteps90', 'jbsteps120', 'jbsteps30pre', 'jbsteps40pre', 'jbsteps60pre']

sdf.columns = cols

print(sdf.columns)

Index(['user.index', 'decision.index.nogap', 'datetime', 'date',
       'sugg.select.slot', 'is.randomized', 'avail', 'send', 'send.active',
       'send.sedentary', 'returned.message', 'response', 'recognized.activity',
       'location_group', 'dec.weather.condition', 'dec.temperature',
       'jbsteps10.zero', 'jbsteps30.zero', 'jbsteps40.zero', 'jbsteps60.zero',
       'jbsteps90.zero', 'jbsteps120.zero', 'jbsteps30pre.zero',
       'jbsteps40pre.zero', 'jbsteps60pre.zero'],
      dtype='str')
Index(['uid', 'decision_idx', 'datetime', 'date', 'day_slot', 'is_randomized',
       'avail', 'send', 'send_active', 'send_sedentary', 'returned_message',
       'response', 'activity', 'location', 'weather', 'temperature',
       'jbsteps10', 'jbsteps30', 'jbsteps40', 'jbsteps60', 'jbsteps90',
       'jbsteps120', 'jbsteps30pre', 'jbsteps40pre', 'jbsteps60pre'],
      dtype='str')


In [140]:
sdf.isnull().sum()
sdf[sdf['temperature'].isnull()].groupby(['uid', 'datetime']).size()

uid  datetime           
2    2015-08-02 22:00:00    1
3    2015-08-06 20:30:00    1
6    2015-08-17 16:00:00    1
7    2015-08-21 22:00:00    1
     2015-08-21 23:30:00    1
                           ..
33   2016-01-01 16:00:00    1
     2016-01-13 14:05:00    1
35   2015-12-15 12:00:00    1
37   2015-12-15 12:00:00    1
     2016-01-13 22:25:00    1
Length: 108, dtype: int64

In [141]:


# df.fillna()
# df.median()
# df.interpolate()

sdf['datetime'] = pd.to_datetime(sdf['datetime'])
sdf = sdf.set_index('datetime')

sdf['temperature'] = sdf.groupby('uid')['temperature'].transform(
    lambda x: x.interpolate(method='time').ffill().bfill()
)

# 3. 恢复索引
sdf = sdf.reset_index()

sdf[sdf['temperature'].isnull()].groupby(['uid', 'datetime']).size()

Series([], dtype: int64)

In [142]:
# 先把 unknown 和报错字符串替换为 NaN
sdf['weather'] = sdf['weather'].replace(
    ['unknown',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.getJSONObject(JSONObject.java:516)',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.<init>(JSONObject.java:179)'],
    pd.NA
)


sdf[sdf['weather'].isnull()].groupby(['uid', 'datetime']).size()

# 用同一用户同一天的众数填补
def fill_mode(x):
    mode = x.mode()
    return x.fillna(mode[0] if len(mode) > 0 else pd.NA)

sdf['weather'] = sdf.groupby(['uid', 'date'])['weather'].transform(fill_mode)


# 剩余的用前向填充
sdf['weather'] = sdf.groupby('uid')['weather'].ffill()

sdf[sdf['weather'].isnull()].groupby(['uid', 'datetime']).size()



sdf.isnull().sum()

datetime               0
uid                    0
decision_idx           0
date                   0
day_slot               0
is_randomized          0
avail                  0
send                   0
send_active            0
send_sedentary         0
returned_message       0
response            4324
activity               0
location               0
weather                0
temperature            0
jbsteps10              0
jbsteps30              0
jbsteps40              0
jbsteps60              0
jbsteps90              0
jbsteps120             0
jbsteps30pre           0
jbsteps40pre           0
jbsteps60pre           0
dtype: int64

In [143]:
'''
send:
 - 0: no send
 - 1: active
 - 2: sedentary

when send.active == False and send.sedentary == False, returned.message is always 'donotnotify'
when send == False, returned.message is always 'donotnotify'
when returned.message == 'donotnotify', send == False
'''
# sdf[sdf['send'] == 0][['send_active', 'send_sedentary']].value_counts()
# sdf[sdf['send_active'] == 0 & (sdf['send_sedentary'] == 0)]['send'].value_counts()
sdf['send'] = sdf['send_active'].astype(int) + sdf['send_sedentary'].astype(int) * 2
sdf['send'].value_counts()

sdf.drop(columns=['send_active', 'send_sedentary'], inplace=True)

sdf.columns

Index(['datetime', 'uid', 'decision_idx', 'date', 'day_slot', 'is_randomized',
       'avail', 'send', 'returned_message', 'response', 'activity', 'location',
       'weather', 'temperature', 'jbsteps10', 'jbsteps30', 'jbsteps40',
       'jbsteps60', 'jbsteps90', 'jbsteps120', 'jbsteps30pre', 'jbsteps40pre',
       'jbsteps60pre'],
      dtype='str')

In [144]:
sdf[sdf['response'].isnull()]['send'].value_counts()
# sdf[sdf['response'].isnull()]['returned_message'].value_counts()


sdf.loc[sdf['response'].isnull() & (sdf['send'] == 0), 'response'] = 'no_send'
sdf[sdf['response'].isnull()]['send'].value_counts()

sdf.loc[sdf['response'].isnull() & (sdf['send'] != 0), 'response'] = 'no_response'

In [145]:
sdf.isnull().sum()

sdf.to_csv('./cleaned_output.csv', index=False)